# 13.7 · 扩散模型 / Diffusion Models (DDPM) ⭐⭐⭐

> **课程定位 / Where this fits**
> 第 7 课，**Part 13 · 生成模型**。**当今图像生成的王者**——DALL·E 2、Stable Diffusion、Midjourney、Sora 全都基于扩散。
> Lesson 7, **Part 13 · Generative Models**. **Today's king of image generation** — DALL·E 2, Stable Diffusion, Midjourney, Sora are all diffusion-based.
>
> 扩散模型的思路出奇简单又强大, 灵感来自物理(墨水在水中扩散)：**前向过程**一步步给真实图像**加噪声**, 直到它变成纯高斯噪声；**反向过程**训练一个网络**一步步去噪**。生成时, 从**纯噪声**出发, 让网络反复去噪, 一张清晰图像就**逐渐浮现**出来。相比 GAN, 扩散**训练稳定、质量和多样性都更好**, 因此取代 GAN 成为主流。本课**从零实现 DDPM**, 在 MNIST 上看"加噪→去噪→生成数字"的全过程。
> The diffusion idea is surprisingly simple yet powerful, inspired by physics (ink diffusing in water): a **forward process** gradually **adds noise** to a real image until it's pure Gaussian noise; a **reverse process** trains a network to **gradually denoise**. To generate, start from **pure noise** and repeatedly denoise — a clear image **gradually emerges**. Versus GANs, diffusion is **stable to train with better quality and diversity**, so it replaced GANs as the mainstream. We **implement DDPM from scratch** and watch the full "add noise → denoise → generate digits" cycle on MNIST.
>
> 💼 **实战/面试视角**：扩散是当下生成式 AI 的**绝对核心**——"前向/反向过程 / 模型预测什么(噪声) / 训练目标 / DDPM vs DDIM 采样 / 为什么比GAN稳" 几乎必考。
> 💼 **Practical/interview angle:** diffusion is the **absolute core** of today's generative AI — "forward/reverse process / what the model predicts (noise) / training objective / DDPM vs DDIM / why more stable than GAN" — almost guaranteed.

> 📐 **符号约定 / Notation**
> - 前向加噪 $x_0 \to x_T$(数据→纯噪声) / forward: data → pure noise
> - $\bar\alpha_t$ —— 累积保留比例, $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$ / cumulative signal level
> - $\epsilon_\theta(x_t, t)$ —— 网络预测的噪声 / network's predicted noise

> 💡 **面试相关 / Interview-relevant**
> - "扩散的前向和反向过程"（出镜率 ★★★★★）
> - "模型到底预测什么(预测噪声ε)"（★★★★★）
> - "训练目标为什么这么简单(MSE预测噪声)"（★★★★）
> - "DDPM vs DDIM(采样速度)"（★★★★）
> - "扩散为什么比GAN稳/质量好"（★★★★★）

---

## 学习目标 / Learning Objectives
1. 理解扩散的前向加噪与反向去噪两个过程。
   Understand diffusion's forward (add noise) and reverse (denoise) processes.
2. 掌握前向过程的闭式公式与"模型预测噪声"的训练目标。
   Master the closed-form forward process and the "predict the noise" objective.
3. **从零实现 DDPM**:训练去噪网络 + 反向采样生成。
   Implement DDPM from scratch: train a denoiser + reverse-sample to generate.
4. 看噪声**逐步去噪成数字**, 理解扩散为何胜过 GAN。
   Watch noise denoise into digits; understand why diffusion beats GANs.

## 目录 / TOC
1. [扩散的核心思想 ⭐](#1)
2. [前向过程:逐步加噪 ⭐](#2)
3. [反向过程:训练去噪 + 生成 ⭐](#3)
4. [逐步去噪可视化 + 对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 扩散的核心思想 ⭐ / The Core Idea of Diffusion

扩散模型有两个相反的过程：
A diffusion model has two opposite processes:
- **前向过程(forward / 扩散)**：拿一张真实图像, **一小步一小步地加高斯噪声**, 经过 $T$ 步(如几百上千步), 图像彻底变成**纯随机噪声**。这个过程**固定、无需学习**(只是按计划加噪)。
  **Forward (diffusion):** take a real image and **add a little Gaussian noise step by step**; after $T$ steps the image becomes **pure random noise**. This process is **fixed, no learning** (just scheduled noising).
- **反向过程(reverse / 去噪)**：训练一个神经网络**学会逆转每一小步**——给一张有噪声的图, 预测并去掉这一步的噪声。这个过程**是要学习的**。
  **Reverse (denoising):** train a neural network to **undo each small step** — given a noisy image, predict and remove that step's noise. This is **what we learn**.

**生成**：训练好后, 从一张**纯噪声**出发(就像前向过程的终点), 让网络**反复去噪 $T$ 步**, 一张清晰的图像就**从噪声里逐渐浮现**。
**Generation:** after training, start from **pure noise** (the forward endpoint) and let the network **denoise for $T$ steps** — a clear image **gradually emerges from the noise**.

**为什么这招高明**(面试)：直接"一步从噪声生成完美图像"太难(GAN 就是硬啃这个, 所以不稳)。扩散把这个超难的任务**拆成几百个极简单的小步**——每步只需"去掉一点点噪声", 容易学、训练稳。这正是它质量高又稳定的根源。
**Why it's clever** (interview): generating "a perfect image from noise in one shot" is too hard (GANs attack this directly, hence instability). Diffusion **breaks it into hundreds of trivial steps** — each just "remove a bit of noise," easy to learn and stable. The root of its quality and stability.


<a id="2"></a>
## 2. 前向过程:逐步加噪 ⭐ / Forward Process: Adding Noise

前向过程每步加一点噪声。妙处(面试点): 虽然定义是"一步步加", 但有一个**闭式公式**能**一步直接算出任意时刻 $t$ 的噪声图** $x_t$(不用真的循环 $t$ 次)：
The forward process adds noise step by step. The beauty (interview): though defined step-wise, a **closed-form formula** gives $x_t$ at **any time $t$ in one shot** (no need to loop $t$ times):

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \qquad \epsilon \sim N(0, I)$$

其中 $\bar\alpha_t$ 随 $t$ 从 1 递减到 ≈0:$t$ 小→$x_t$ 几乎是原图($\sqrt{\bar\alpha_t}\approx1$);$t$ 大→$x_t$ 几乎是纯噪声。这让训练可以**随机抽一个 $t$ 直接造出对应的噪声图**, 高效。
where $\bar\alpha_t$ decreases from 1 to ≈0 as $t$ grows: small $t$ → $x_t$ ≈ the original; large $t$ → $x_t$ ≈ pure noise. This lets training **sample a random $t$ and directly construct the corresponding noisy image**, efficiently.

下面用一个数字, 可视化它在不同 $t$ 被加噪的样子(从清晰 → 纯噪声)。
Below we visualize one digit at various $t$ (clear → pure noise).


In [ ]:
import os, math, numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white"); torch.manual_seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])
train = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tfm)
loader = DataLoader(Subset(train, range(12000)), batch_size=128, shuffle=True)

# 噪声调度(noise schedule): beta 线性增大 / linear beta schedule
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                          # ᾱ_t = 累积 α / cumulative product

def add_noise(x0, t, eps):
    """前向: 一步算出 t 时刻的噪声图 x_t / forward: closed-form x_t."""
    return torch.sqrt(abar[t])[:,None,None,None]*x0 + torch.sqrt(1-abar[t])[:,None,None,None]*eps

# 可视化一个数字在不同 t 的加噪过程 / visualize one digit noised at increasing t
x0 = train[7][0].unsqueeze(0)
ts = [0, 25, 50, 100, 150, 199]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for ax, t in zip(axes, ts):
    xt = add_noise(x0, torch.tensor([t]), torch.randn_like(x0))
    ax.imshow(xt[0,0], cmap="gray"); ax.set_title(f"t={t}\nᾱ={abar[t]:.2f}", fontsize=9); ax.axis("off")
fig.suptitle("前向过程: 逐步加噪 (t=0 清晰原图 → t=199 纯噪声); ᾱ 是'还剩多少原图信息'")
plt.tight_layout(); plt.show()
print("前向是固定的(按schedule加噪); 闭式公式 x_t=√ᾱ·x₀+√(1-ᾱ)·ε 可一步算出任意t的噪声图")


<a id="3"></a>
## 3. 反向过程:训练去噪 + 生成 ⭐ / Reverse: Train Denoiser & Generate

**核心洞察(面试必考): 模型预测的是"噪声 $\epsilon$", 不是图像**。训练时：随机取一张图 $x_0$、随机取一个时刻 $t$、随机取噪声 $\epsilon$, 用闭式公式造出 $x_t$；让网络 $\epsilon_\theta(x_t, t)$ **预测当初加进去的噪声 $\epsilon$**。损失就是最简单的 **MSE**:
**Key insight (must-know): the model predicts the "noise $\epsilon$", not the image.** Training: take a random image $x_0$, a random time $t$, random noise $\epsilon$, build $x_t$ via the closed form; have the network $\epsilon_\theta(x_t, t)$ **predict the noise $\epsilon$ that was added**. The loss is the simplest **MSE**:

$$\mathcal{L} = \mathbb{E}_{x_0, t, \epsilon}\big[\,\|\epsilon - \epsilon_\theta(x_t, t)\|^2\,\big]$$

就这么简单！(去噪扩散的训练目标比 GAN/VAE 都简单纯粹——只是回归噪声。)
That's it! (The training objective is simpler/purer than GAN/VAE — just regress the noise.)

**生成(反向采样)**：从 $x_T \sim N(0,I)$ 纯噪声开始, 对 $t = T-1, \dots, 0$ 反复：用网络预测噪声、去掉一点、(加一点随机性), 得到稍微干净的 $x_{t-1}$。$T$ 步后得到生成图 $x_0$。
**Generation (reverse sampling):** start from $x_T \sim N(0,I)$; for $t = T-1,\dots,0$: predict the noise, remove a bit, (add a little randomness), getting a slightly cleaner $x_{t-1}$. After $T$ steps, the generated $x_0$.

网络需要**知道当前是第几步 $t$**(去噪强度随 $t$ 不同), 所以把 $t$ 编码成向量(类似位置编码)喂进网络。下面从零实现并训练。
The network must **know the current step $t$** (denoising strength varies with $t$), so we encode $t$ into a vector (like positional encoding) fed into the net. We implement and train from scratch.


In [ ]:
def time_embedding(t, dim=64):
    """把时间步 t 编码成向量(正弦式, 类似Transformer位置编码) / sinusoidal time embedding."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

class Denoiser(nn.Module):
    """预测噪声的小 CNN, 把时间嵌入加到每层特征图 / small CNN predicting noise, with time embedding."""
    def __init__(self, c=32):
        super().__init__()
        self.tmlp = nn.Sequential(nn.Linear(64, c), nn.ReLU(), nn.Linear(c, c))   # 时间嵌入→通道偏置 / time→bias
        self.e1 = nn.Conv2d(1, c, 3, padding=1); self.e2 = nn.Conv2d(c, c, 3, padding=1)
        self.mid = nn.Conv2d(c, c, 3, padding=1)
        self.d2 = nn.Conv2d(c, c, 3, padding=1); self.d1 = nn.Conv2d(c, 1, 3, padding=1)
    def forward(self, x, t):
        te = self.tmlp(time_embedding(t))[:, :, None, None]    # 时间信息广播加到特征图 / broadcast time info
        h = F.relu(self.e1(x) + te); h = F.relu(self.e2(h) + te)
        h = F.relu(self.mid(h) + te); h = F.relu(self.d2(h) + te)
        return self.d1(h)                                      # 输出预测的噪声(和输入同尺寸) / predicted noise

torch.manual_seed(0); net = Denoiser(); opt = torch.optim.Adam(net.parameters(), 1e-3)
t0 = time.time(); losses = []
for epoch in range(12):
    for xb, _ in loader:
        b = xb.size(0)
        t = torch.randint(0, T, (b,))                          # 随机时刻 / random timestep
        eps = torch.randn_like(xb)                             # 随机噪声 / random noise
        xt = add_noise(xb, t, eps)                             # 造噪声图 / build noisy image
        loss = F.mse_loss(net(xt, t), eps)                     # 让网络预测这个噪声 / predict the noise
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
print(f"DDPM 去噪网络训练完成 ({time.time()-t0:.0f}s), 最终噪声预测 MSE = {losses[-1]:.3f}")

@torch.no_grad()
def sample(n=8, keep=None):
    """反向采样: 从纯噪声逐步去噪生成图 / reverse sampling: pure noise → image."""
    x = torch.randn(n, 1, 28, 28); traj = {}
    for i in reversed(range(T)):
        t = torch.full((n,), i)
        eps = net(x, t)                                        # 预测噪声 / predict noise
        a, ab = alphas[i], abar[i]
        x = (1/torch.sqrt(a)) * (x - (1-a)/torch.sqrt(1-ab) * eps)   # 去掉一步噪声 / remove one step
        if i > 0: x = x + torch.sqrt(betas[i]) * torch.randn_like(x) # 加一点随机性 / add randomness
        if keep and i in keep: traj[i] = x.clone()
    return x, traj
gen, _ = sample(16)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16): axes[i//8,i%8].imshow(gen[i,0], cmap="gray"); axes[i//8,i%8].axis("off")
fig.suptitle("DDPM 生成: 从纯噪声反向去噪 200 步 → 生成的数字"); plt.tight_layout(); plt.show()
print("生成的数字(从纯噪声去噪而来); 训练目标极简(MSE预测噪声), 却能生成多样清晰的样本")


<a id="4"></a>
## 4. 逐步去噪可视化 + 对比 + 小结 ⭐ / Step-by-Step Denoising & Comparison

扩散最直观的地方：**看一张图如何从纯噪声一步步"显影"成数字**。我们保存采样过程中的若干中间步骤。
The most intuitive part: **watch an image "develop" from pure noise into a digit step by step.** We save intermediate steps during sampling.


In [ ]:
# 可视化反向去噪轨迹: 从纯噪声 → 逐步清晰 / visualize the reverse denoising trajectory
torch.manual_seed(3)
_, traj = sample(n=6, keep=[199, 150, 100, 60, 30, 0])
steps = [199, 150, 100, 60, 30, 0]
fig, axes = plt.subplots(6, 6, figsize=(9, 9))
for col, t in enumerate(steps):
    for row in range(6):
        axes[row, col].imshow(traj[t][row,0], cmap="gray"); axes[row, col].axis("off")
    axes[0, col].set_title(f"t={t}", fontsize=10)
fig.suptitle("反向去噪轨迹(每列一个时刻, 从左纯噪声→右清晰): 数字逐步'显影'", y=1.0)
plt.tight_layout(); plt.show()
print("从左(t=199纯噪声)到右(t=0), 网络一步步去噪, 数字逐渐浮现 → 这就是扩散生成的全过程")


**扩散 vs GAN/VAE**(面试高频)与**实务**：
**Diffusion vs GAN/VAE** (high-frequency) and practice:
- **vs GAN**:扩散**训练稳定**(就是个回归任务, 没有对抗博弈/模式坍塌)、**质量和多样性更好**——所以取代 GAN 成为主流。**代价: 采样慢**(要跑 $T$ 步去噪, GAN 只需一次前向)。
  **vs GAN:** diffusion is **stable to train** (just regression, no adversarial game/mode collapse) with **better quality and diversity** — hence it replaced GANs. **Cost: slow sampling** ($T$ denoising steps vs GAN's single forward).
- **加速采样: DDIM**——一种确定性采样, 能**跳步**(如只用 20~50 步而非 1000 步), 大幅加速, 质量损失很小。是实用部署的关键。
  **Faster sampling: DDIM** — a deterministic sampler that **skips steps** (e.g. 20–50 instead of 1000), much faster with little quality loss. Key for deployment.
- **预测噪声 vs 预测图像**:实践证明**预测噪声 $\epsilon$** 效果最好(也可预测 $x_0$ 或 v-prediction, 数学等价但训练动态不同)。
  **Predict noise vs image:** predicting the **noise $\epsilon$** works best in practice (predicting $x_0$ or v-prediction are equivalent but train differently).

```
扩散两过程: 前向(固定)逐步加噪 数据→纯噪声; 反向(学习)逐步去噪 噪声→数据
关键洞察: 把'一步生成完美图'(GAN硬啃→不稳)拆成几百个'去一点噪'的简单小步→易学/稳定
前向闭式: x_t=√ᾱ_t·x₀+√(1-ᾱ_t)·ε, 一步算任意t; 训练随机抽t造噪声图
训练目标(极简): 网络ε_θ(x_t,t)预测加进去的噪声ε, 损失=MSE(ε, ε_θ); 网络需知道t(时间嵌入)
生成: x_T~N(0,1)→反复预测噪声去噪T步→x_0; 可视化=数字从噪声逐步'显影'
vs GAN: 稳定+质量多样性更好(取代GAN); 代价采样慢(T步) → DDIM跳步加速
应用: DALL·E2/Stable Diffusion/Midjourney/Sora 全基于扩散
```

### 💡 面试速查 / Interview cheat-sheet
1. **两过程**: 前向固定加噪(数据→噪声), 反向学习去噪(噪声→数据)。
   Two processes: fixed forward noising, learned reverse denoising.
2. **预测噪声**: 网络预测每步加的噪声ε; 损失=MSE(极简)。
   Predict noise: the net predicts each step's noise ε; loss = MSE (very simple).
3. **为什么稳/好**: 把难题拆成几百简单小步(回归,无对抗)→稳定+高质量多样。
   Why stable/good: breaks the task into many easy steps (regression, no adversary).
4. **vs GAN**: 更稳更好但采样慢(T步); DDIM跳步加速。
   vs GAN: more stable/better but slow sampling; DDIM speeds it up.
5. **应用**: DALL·E2/SD/Midjourney/Sora 全基于扩散。
   Apps: DALL·E2/SD/Midjourney/Sora all diffusion-based.

### 下一节 / Next
**13.8 Stable Diffusion 原理**——把扩散(13.7)、自编码器(13.1)、CLIP(10.10/12.4)拼起来, 就得到了改变世界的**文生图**模型。它的三大关键: ①在**潜空间**而非像素空间扩散(快得多) ②用 **U-Net** 做去噪网络 ③用 **CLIP 文本编码**做条件。我们会讲清这套架构怎么协同工作。
**13.8 Stable Diffusion Internals** — combine diffusion (13.7), autoencoders (13.1), and CLIP (10.10/12.4) and you get the world-changing **text-to-image** model. Its three keys: ① diffuse in **latent space** not pixels (much faster) ② a **U-Net** denoiser ③ **CLIP text** conditioning. We'll explain how this architecture works together.
